# 08.3 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/08-deep-learning/03-activations-and-losses/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"        # hide TensorFlow start-up messages
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

def sigmoid(z): return 1 / (1 + np.exp(-z))
def relu(z):    return np.maximum(0, z)
def softmax(z):
    e = np.exp(z - np.max(z))          # subtracting the max avoids overflow, same result
    return e / e.sum()

import keras
from keras import layers
keras.utils.set_random_seed(42)      # same starting weights every run (small differences across machines are normal)
print("Keras", keras.__version__, "on", keras.backend.backend())

from sklearn.datasets import make_moons
X, y = make_moons(600, noise=0.2, random_state=42)

def moons_accuracy(activation, hidden_layers=2):
    keras.utils.set_random_seed(42)
    model = keras.Sequential([keras.Input((2,))] +
                             [layers.Dense(16, activation=activation) for _ in range(hidden_layers)] +
                             [layers.Dense(1, activation="sigmoid")])
    model.compile(optimizer=keras.optimizers.Adam(0.01), loss="binary_crossentropy", metrics=["accuracy"])
    history = model.fit(X, y, epochs=100, batch_size=32, verbose=0)
    return model, history.history["accuracy"][-1]

Keras 3.15.1 on tensorflow


## Exercise 1

Equal scores share equally: [1/3, 1/3, 1/3]. With [10, 0, 0], $e^{10} = 22{,}026$ dwarfs $e^0 = 1$, so the first class gets almost everything. Softmax depends only on **differences** between scores; large gaps produce near-certainty.

In [2]:
print(softmax(np.array([1.0, 1.0, 1.0])).round(4))
print(softmax(np.array([10.0, 0.0, 0.0])).round(5))

[0.3333 0.3333 0.3333]
[9.9991e-01 5.0000e-05 5.0000e-05]


## Exercise 2

$-\ln(1 - 0.9) = -\ln 0.1 = 2.303$: exactly as bad as predicting 0.1 for a churner. The loss is symmetric: confidence in the wrong direction is what is punished.

In [3]:
print(-np.log(1 - 0.9), float(keras.losses.BinaryCrossentropy()(np.array([[0.0]]), np.array([[0.9]]))))

2.302585092994046 2.3025848865509033


## Exercise 3

In [4]:
from sklearn.linear_model import LogisticRegression
_, acc4 = moons_accuracy(None, hidden_layers=4)
print(f"4 linear hidden layers: {acc4:.1%}   logistic regression: {LogisticRegression().fit(X, y).score(X, y):.1%}")

4 linear hidden layers: 87.7%   logistic regression: 87.5%


About 87.7%, within a fraction of a point of logistic regression's 87.5%. Still a straight line: four linear layers are one linear layer. The tiny gap is only because gradient descent and scikit-learn settle on slightly different straight lines.

## Exercise 4

- **(a)** `Dense(1)` (linear) with MSE, or MAE if a few huge orders should not dominate.
- **(b)** `Dense(5, activation="softmax")` with sparse categorical cross-entropy (labels 0–4).
- **(c)** `Dense(3, activation="sigmoid")` with binary cross-entropy: each tag is its own yes/no, and a photo can have several.